# Phase 2: Segmentación Dinámica Forex

Este notebook recorta *trozos* del gráfico con dos enfoques:
1. **Heurística de picos/valles** para extraer impulsos (ej. 15 pips) y rangos (ej. 40 pips).
2. **Matrix Profile** (motifs/discords) si `stumpy` está instalado.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from IPython.display import display

try:
    import stumpy
    HAS_STUMPY = True
except Exception:
    HAS_STUMPY = False

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
print("HAS_STUMPY =", HAS_STUMPY)


In [ ]:
# --- Parámetros principales ---
DATA_PATH = "./data/parquet_h1/USDJPY.parquet"
SYMBOL = "USDJPY"

SEGMENT_METHOD = "heuristic"  # 'heuristic' o 'matrix_profile'

# Heurística pivots/legs
PIVOT_PROMINENCE_PIPS = 6
MIN_IMPULSE_PIPS = 15
MAX_RANGE_PIPS = 40
MIN_RANGE_BARS = 8
MAX_RANGE_BARS = 64

# Matrix Profile
MATRIX_WINDOW_BARS = 48
TOP_K_MATRIX_SEGMENTS = 12

# Visualización
PLOT_LAST_BARS = 1200


In [ ]:
def pip_size(symbol: str) -> float:
    s = symbol.upper()
    return 0.01 if s.endswith("JPY") else 0.0001

def load_ohlc(path: str) -> pd.DataFrame:
    df = pd.read_parquet(path).copy()
    if "timestamp" not in df.columns or "close" not in df.columns:
        raise ValueError("Se requieren columnas timestamp y close")
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp", "close"])
    df = df.sort_values("timestamp").drop_duplicates(subset=["timestamp"], keep="last")
    df = df.reset_index(drop=True)
    return df

def find_pivot_indices(close: np.ndarray, prominence_price: float) -> np.ndarray:
    peaks, _ = find_peaks(close, prominence=prominence_price)
    troughs, _ = find_peaks(-close, prominence=prominence_price)
    pivots = np.unique(np.concatenate(([0], peaks, troughs, [len(close)-1])))
    pivots.sort()
    return pivots

def extract_impulses(df: pd.DataFrame, pivot_idx: np.ndarray, pip: float, min_impulse_pips: float) -> pd.DataFrame:
    close = df["close"].to_numpy(dtype=float)
    ts = df["timestamp"].to_numpy()
    rows = []
    for a, b in zip(pivot_idx[:-1], pivot_idx[1:]):
        move_pips = (close[b] - close[a]) / pip
        if abs(move_pips) >= min_impulse_pips:
            rows.append({
                "segment_type": "impulse",
                "direction": "up" if move_pips > 0 else "down",
                "start_idx": int(a),
                "end_idx": int(b),
                "start_ts": ts[a],
                "end_ts": ts[b],
                "n_bars": int(b - a + 1),
                "move_pips": float(move_pips),
                "range_pips": float((df.loc[a:b, "high"].max() - df.loc[a:b, "low"].min()) / pip) if {"high","low"}.issubset(df.columns) else float(abs(move_pips)),
            })
    return pd.DataFrame(rows)

def extract_ranges(df: pd.DataFrame, pip: float, max_range_pips: float, min_bars: int, max_bars: int) -> pd.DataFrame:
    close = df["close"].to_numpy(dtype=float)
    ts = df["timestamp"].to_numpy()
    n = len(df)
    rows = []
    i = 0
    while i <= n - min_bars:
        best_end = None
        best_span = None
        max_w = min(max_bars, n - i)
        for w in range(min_bars, max_w + 1):
            j = i + w
            span_pips = (close[i:j].max() - close[i:j].min()) / pip
            if span_pips <= max_range_pips:
                best_end = j - 1
                best_span = span_pips
            else:
                break

        if best_end is not None:
            rows.append({
                "segment_type": "range",
                "direction": "flat",
                "start_idx": int(i),
                "end_idx": int(best_end),
                "start_ts": ts[i],
                "end_ts": ts[best_end],
                "n_bars": int(best_end - i + 1),
                "move_pips": float((close[best_end] - close[i]) / pip),
                "range_pips": float(best_span),
            })
            i = best_end + 1
        else:
            i += 1

    return pd.DataFrame(rows)

def extract_matrix_profile_segments(df: pd.DataFrame, m: int, top_k: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not HAS_STUMPY:
        raise RuntimeError("Matrix Profile requiere stumpy. Instala con: pip install stumpy")
    close = df["close"].to_numpy(dtype=np.float64)
    ts = df["timestamp"].to_numpy()
    mp = stumpy.stump(close, m=m)
    profile = mp[:, 0].astype(float)

    motif_idx = np.argsort(profile)[:top_k]
    discord_idx = np.argsort(profile)[-top_k:][::-1]

    def build_rows(indices, kind):
        rows = []
        for i in indices:
            s = int(i)
            e = int(i + m - 1)
            rows.append({
                "segment_type": f"matrix_{kind}",
                "direction": "na",
                "start_idx": s,
                "end_idx": e,
                "start_ts": ts[s],
                "end_ts": ts[e],
                "n_bars": int(m),
                "move_pips": float('nan'),
                "range_pips": float('nan'),
                "matrix_profile": float(profile[i]),
            })
        return pd.DataFrame(rows)

    return build_rows(motif_idx, "motif"), build_rows(discord_idx, "discord")


In [ ]:
df = load_ohlc(DATA_PATH)
pip = pip_size(SYMBOL)

print(f"Filas: {len(df):,} | desde {df['timestamp'].min()} hasta {df['timestamp'].max()}")
print(f"Pip size ({SYMBOL}): {pip}")

segments_impulse = pd.DataFrame()
segments_range = pd.DataFrame()
segments_motif = pd.DataFrame()
segments_discord = pd.DataFrame()

if SEGMENT_METHOD == "heuristic":
    pivots = find_pivot_indices(df["close"].to_numpy(dtype=float), PIVOT_PROMINENCE_PIPS * pip)
    segments_impulse = extract_impulses(df, pivots, pip, MIN_IMPULSE_PIPS)
    segments_range = extract_ranges(df, pip, MAX_RANGE_PIPS, MIN_RANGE_BARS, MAX_RANGE_BARS)

    print(f"Pivots detectados: {len(pivots)}")
    print(f"Impulsos (>= {MIN_IMPULSE_PIPS} pips): {len(segments_impulse)}")
    print(f"Rangos (<= {MAX_RANGE_PIPS} pips): {len(segments_range)}")

    display(segments_impulse.head(10))
    display(segments_range.head(10))

elif SEGMENT_METHOD == "matrix_profile":
    segments_motif, segments_discord = extract_matrix_profile_segments(df, MATRIX_WINDOW_BARS, TOP_K_MATRIX_SEGMENTS)
    print(f"Motifs: {len(segments_motif)} | Discords: {len(segments_discord)}")
    display(segments_motif.head(10))
    display(segments_discord.head(10))

else:
    raise ValueError("SEGMENT_METHOD debe ser 'heuristic' o 'matrix_profile'")


In [ ]:
plot_df = df.iloc[-min(PLOT_LAST_BARS, len(df)):].copy()
offset = int(plot_df.index[0])

plt.figure(figsize=(16, 6))
plt.plot(plot_df["timestamp"], plot_df["close"], lw=1.2, label="close", color="black")

if SEGMENT_METHOD == "heuristic":
    if len(segments_range):
        for _, r in segments_range.iterrows():
            if r.start_idx < offset:
                continue
            plt.axvspan(r.start_ts, r.end_ts, color="steelblue", alpha=0.12)

    if len(segments_impulse):
        for _, r in segments_impulse.iterrows():
            if r.start_idx < offset:
                continue
            c = "green" if r.direction == "up" else "red"
            s = int(r.start_idx)
            e = int(r.end_idx)
            plt.plot([df.loc[s, "timestamp"], df.loc[e, "timestamp"]], [df.loc[s, "close"], df.loc[e, "close"]], color=c, lw=2.0, alpha=0.9)

elif SEGMENT_METHOD == "matrix_profile":
    if len(segments_motif):
        for _, r in segments_motif.iterrows():
            if r.start_idx < offset:
                continue
            plt.axvspan(r.start_ts, r.end_ts, color="green", alpha=0.18)
    if len(segments_discord):
        for _, r in segments_discord.iterrows():
            if r.start_idx < offset:
                continue
            plt.axvspan(r.start_ts, r.end_ts, color="red", alpha=0.15)

plt.title(f"Segmentación dinámica ({SEGMENT_METHOD}) - {SYMBOL}")
plt.xlabel("timestamp")
plt.ylabel("price")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# Export opcional
OUTPUT_DIR = "./phases/phase02_market_alphabet/output_segments"
from pathlib import Path
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

if SEGMENT_METHOD == "heuristic":
    if len(segments_impulse):
        segments_impulse.to_parquet(f"{OUTPUT_DIR}/{SYMBOL}_impulses.parquet", index=False)
    if len(segments_range):
        segments_range.to_parquet(f"{OUTPUT_DIR}/{SYMBOL}_ranges.parquet", index=False)
    print("Export heuristic listo")

elif SEGMENT_METHOD == "matrix_profile":
    if len(segments_motif):
        segments_motif.to_parquet(f"{OUTPUT_DIR}/{SYMBOL}_matrix_motifs.parquet", index=False)
    if len(segments_discord):
        segments_discord.to_parquet(f"{OUTPUT_DIR}/{SYMBOL}_matrix_discords.parquet", index=False)
    print("Export matrix_profile listo")
